# IBKR API notebook

#### Connection

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from ib_async import *
import pandas as pd
util.startLoop()
global historical_data_interval, duration

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)
if ib.isConnected():
    print("✅ Connected to IBKR API")
else:
    print("❌Failed to connect to IBKR API")
util.logToConsole(logging.INFO)

2025-05-05 17:04:42,012 - INFO - Connecting to 127.0.0.1:7497 with clientId 14...
2025-05-05 17:04:42,013 - INFO - Connected
2025-05-05 17:04:42,015 - INFO - Logged on to server version 178
2025-05-05 17:04:42,056 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfarm.nj
2025-05-05 17:04:42,057 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfuture
2025-05-05 17:04:42,058 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:eufarm
2025-05-05 17:04:42,058 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:cashfarm
2025-05-05 17:04:42,058 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfarm
2025-05-05 17:04:42,059 - INFO - Warning 2106, reqId -1: La connexion de donn\u00e9es HMDS est OK:euhmds
2025-05-05 17:04:42,059 - INFO - Warning 2106, reqId -1: La connexion de donn\u00e9es HMDS est OK:ushmds
2025-05-05 17:04:4

✅ Connected to IBKR API


2025-05-05 17:07:16,499 - INFO - Warning 2174, reqId 5: Avertissement: You submitted request with date-time attributes without explicit time zone. Please switch to use yyyymmdd-hh:mm:ss in UTC or use instrument time zone, like US/Eastern. Implied time zone functionality will be removed in the next API release, contract: Index(symbol='NDX', exchange='NASDAQ', currency='USD')


## Request Historical data

#### Choose your contract

In [3]:
#contract = CFD('IBUST100', 'SMART', 'USD')
#contract = Forex(pair="EURUSD", exchange='IDEALPRO')
contract = Index('NDX', 'NASDAQ', 'USD')
contract_details = ib.reqContractDetails(contract)
# Extract and display the desired fields from contract_details
filtered_details = [
    {
        "secType": detail.contract.secType,
        "conId": detail.contract.conId,
        "symbol": detail.contract.symbol,
        "exchange": detail.contract.exchange,
        "longName": detail.longName,
        "timezoneId": detail.timeZoneId,
        "tradingHours": "\n".join(
            [f"  {segment}" for segment in detail.tradingHours.split(";")]
        ),
        "liquidHours": "\n".join(
            [f"  {segment}" for segment in detail.liquidHours.split(";")]
        ),
        "minSize": detail.minSize,
    }
    for detail in contract_details
]

# Print the filtered details in a clear format
for idx, detail in enumerate(filtered_details, start=1):
    print(f"Contract Detail {idx}:")
    for key, value in detail.items():
        print(f"  {key}: {value}")
    print()

Contract Detail 1:
  secType: IND
  conId: 416843
  symbol: NDX
  exchange: NASDAQ
  longName: NASDAQ 100 Stock Index
  timezoneId: US/Eastern
  tradingHours:   20250505:0930-20250505:1600
  20250506:0930-20250506:1600
  20250507:0930-20250507:1600
  20250508:0930-20250508:1600
  20250509:0930-20250509:1600
  liquidHours:   20250505:0930-20250505:1600
  20250506:0930-20250506:1600
  20250507:0930-20250507:1600
  20250508:0930-20250508:1600
  20250509:0930-20250509:1600
  minSize: 1.0



#### Check first data timestamp available

In [ ]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='TRADES', useRTH=False)
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
logging.info(f"First date of data available: {formatted_time}")

### Request historical data function

**End date choice**

In [ ]:
save_path = "./database/AAPL_10secs_20220131_to_20250403.parquet"
# Load the parquet file into a DataFrame
retrieved_df = pd.read_parquet(save_path)
# Retrieve the first date from your existing DataFrame
first_date = retrieved_df.iloc[0]['date']  # Assuming 'date' is the column name
end_date = first_date.strftime('%Y%m%d %H:%M:%S')  # Format as 'yyyyMMdd HH:mm:ss'
logging.info(f"First date in the DataFrame: {first_date}")
logging.info(f"End date: {end_date}")

In [6]:
# yesterday's date
# yesterday's date
end_date = (pd.Timestamp.now(tz='UTC') - pd.DateOffset(days=1)).strftime('%Y%m%d %H:%M:%S')

In [ ]:
#today's date
end_date = pd.Timestamp.now(tz='UTC').strftime('%Y%m%d-%H:%M:%S')

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`.  
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  

In [7]:
historical_data_interval = '10 secs' 
request_duration = '1 M'  # Duration in days (use D, not "day")
price_source = 'TRADES'  # 'BID', 'ASK', or 'TRADES'
bars = ib.reqHistoricalData(
        contract,
        endDateTime=end_date,
        durationStr=str(request_duration),
        barSizeSetting=str(historical_data_interval),
        whatToShow=price_source,
        useRTH=False, 
        formatDate=2,
        timeout = 0)

Convert the list of bars to a data frame and print the first and last rows:

In [8]:
bars[0]
df = util.df(bars)
print("DataFrame shape:", df.shape)

display(df.head(n=20))
display(df.tail(n=20))

DataFrame shape: (49119, 8)


,date,open,high,low,close,volume,average,barCount
0,2025-04-03 13:30:10+00:00,18774.15,18778.98,18774.15,18778.98,0.0,0.0,5
1,2025-04-03 13:30:20+00:00,18777.58,18787.97,18777.58,18787.97,0.0,0.0,10
2,2025-04-03 13:30:30+00:00,18790.09,18805.48,18790.09,18805.48,0.0,0.0,10
3,2025-04-03 13:30:40+00:00,18801.07,18815.94,18799.58,18815.94,0.0,0.0,10
4,2025-04-03 13:30:50+00:00,18825.30,18826.80,18820.67,18825.26,0.0,0.0,10
5,2025-04-03 13:31:00+00:00,18829.01,18832.79,18821.42,18832.79,0.0,0.0,10
6,2025-04-03 13:31:10+00:00,18833.24,18840.19,18833.24,18836.74,0.0,0.0,10
7,2025-04-03 13:31:20+00:00,18835.19,18835.39,18816.84,18819.47,0.0,0.0,10
8,2025-04-03 13:31:30+00:00,18814.60,18827.28,18808.63,18827.28,0.0,0.0,10
9,2025-04-03 13:31:40+00:00,18827.97,18832.26,18820.79,18827.36,0.0,0.0,10


,date,open,high,low,close,volume,average,barCount
49099,2025-05-02 19:56:40+00:00,20092.33,20093.24,20091.55,20092.71,0.0,0.0,10
49100,2025-05-02 19:56:50+00:00,20093.07,20093.10,20090.59,20092.08,0.0,0.0,10
49101,2025-05-02 19:57:00+00:00,20095.48,20101.30,20095.48,20101.30,0.0,0.0,10
49102,2025-05-02 19:57:10+00:00,20101.86,20105.15,20101.24,20101.76,0.0,0.0,10
49103,2025-05-02 19:57:20+00:00,20102.92,20102.92,20098.87,20100.36,0.0,0.0,10
49104,2025-05-02 19:57:30+00:00,20102.25,20103.69,20102.24,20102.30,0.0,0.0,10
49105,2025-05-02 19:57:40+00:00,20102.44,20103.46,20102.43,20102.43,0.0,0.0,10
49106,2025-05-02 19:57:50+00:00,20101.99,20103.06,20100.03,20103.06,0.0,0.0,10
49107,2025-05-02 19:58:00+00:00,20101.78,20102.18,20092.94,20095.13,0.0,0.0,10
49108,2025-05-02 19:58:10+00:00,20095.76,20098.78,20095.69,20096.58,0.0,0.0,10


#### Checking if volume and average columns are empty or not and remove it if empty

In [9]:
# Remove the 'volume', 'average', and 'barCount' columns from the DataFrame
df = df.drop(columns=['volume', 'average', 'barCount'])

# Display the updated DataFrame
display(df.head())

,date,open,high,low,close
0,2025-04-03 13:30:10+00:00,18774.15,18778.98,18774.15,18778.98
1,2025-04-03 13:30:20+00:00,18777.58,18787.97,18777.58,18787.97
2,2025-04-03 13:30:30+00:00,18790.09,18805.48,18790.09,18805.48
3,2025-04-03 13:30:40+00:00,18801.07,18815.94,18799.58,18815.94
4,2025-04-03 13:30:50+00:00,18825.30,18826.80,18820.67,18825.26


Construction du nom du fichier et sauvegarde en `.parquet` dans le dossier `database`

Structure du nom du fichier : `Symbol_Interval_StartDate_EndDate_PriceSource.parquet`

In [10]:
# Récupérer la devise et l'unité de temps
symbol = contract.symbol

# Construire le nom du fichier
start_date = df['date'].iloc[0].strftime('%Y%m%d')
end_date = df['date'].iloc[-1].strftime('%Y%m%d')
# Structure
save_path = f"../marketData/{symbol}_{historical_data_interval.replace(' ', '')}_{start_date}_to_{end_date}_{price_source}.parquet"

# Sauvegarder le DataFrame en fichier parquet
df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")

Fichier sauvegardé sous le nom : ../marketData/NDX_10secs_20250403_to_20250502_TRADES.parquet


In [14]:
import pandas as pd
save_path = "../marketData/NDX_10secs_20220214_to_20250502_TRADES.parquet"
#save_path = "../marketData/NDX_10secs_20220131_to_20250403.parquet"

# Load the parquet file into a DataFrame

retrieved_df = pd.read_parquet(save_path)
retrieved_df['date'] = retrieved_df['date'].dt.tz_convert('Europe/Paris')
print(type(retrieved_df['date'].iloc[0]))

# Display the first and last rows of the DataFrame
display(retrieved_df.head())
display(retrieved_df.tail())

<class 'pandas._libs.tslibs.timestamps.Timestamp'>


,date,open,high,low,close
0,2025-04-03 15:30:10+02:00,18774.15,18778.98,18774.15,18778.98
1,2025-04-03 15:30:20+02:00,18777.58,18787.97,18777.58,18787.97
2,2025-04-03 15:30:30+02:00,18790.09,18805.48,18790.09,18805.48
3,2025-04-03 15:30:40+02:00,18801.07,18815.94,18799.58,18815.94
4,2025-04-03 15:30:50+02:00,18825.30,18826.80,18820.67,18825.26


,date,open,high,low,close
49114,2025-05-02 21:59:10+02:00,20092.85,20099.47,20092.85,20099.47
49115,2025-05-02 21:59:20+02:00,20100.19,20106.53,20100.19,20106.53
49116,2025-05-02 21:59:30+02:00,20107.10,20110.80,20103.41,20110.80
49117,2025-05-02 21:59:40+02:00,20111.43,20112.79,20103.82,20103.82
49118,2025-05-02 21:59:50+02:00,20106.25,20106.56,20097.16,20098.05


## Additional features

#### Data pre processing

In [ ]:
import os
import pandas as pd
from igtrader.Strategies.Helpers import load_data, resample_ohlc

In [ ]:
symbol = 'NDX'
interval = '10secs'
start_date = '20220214'
end_date = '20250411'
source = 'TRADES'

save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}_{source}.parquet"
df = pd.read_parquet(save_path, engine='pyarrow')
df

#### DataFrame jointure

In [12]:
import pandas as pd

symbol = 'NDX'
interval = '10secs'
start_date = '20220214'
end_date = '20250502'
price_source= 'TRADES'

# 1. Load the existing dataframe
existing_file_path = "../marketData/NDX_10secs_20220214_to_20250411_TRADES.parquet"
existing_df = pd.read_parquet(existing_file_path)

# 3. Check for the last timestamp in existing data
last_timestamp = existing_df['date'].max()
print(f"Last timestamp in existing data: {last_timestamp}")

# 4. Filter the new dataframe to keep only data after the last timestamp
updated_df_path = "../marketData/NDX_10secs_20250403_to_20250502_TRADES.parquet"
updated_df = pd.read_parquet(updated_df_path)
updated_df = df[df['date'] > last_timestamp]
print(f"New data points to be added: {len(updated_df)}")

# 5. Concatenate the dataframes
merged_df = pd.concat([existing_df, updated_df])

# 6. Sort the merged dataframe by date
merged_df = merged_df.sort_values('date')

# 7. Reset the index to create a clean sequential index
merged_df = merged_df.reset_index(drop=True)

# Display summary
print(f"Original data points: {len(existing_df)}")
print(f"New data points: {len(updated_df)}")
print(f"Total data points after merge: {len(merged_df)}")
merged_df

Last timestamp in existing data: 2025-04-11 19:59:50+00:00
New data points to be added: 32746
Original data points: 1848348
New data points: 32746
Total data points after merge: 1881094


,date,open,high,low,close
0,2022-02-14 14:30:10+00:00,14233.40,14239.47,14233.40,14239.47
1,2022-02-14 14:30:20+00:00,14239.47,14242.25,14231.47,14235.56
2,2022-02-14 14:30:30+00:00,14235.56,14242.16,14235.56,14239.24
3,2022-02-14 14:30:40+00:00,14239.24,14239.45,14233.44,14236.41
4,2022-02-14 14:30:50+00:00,14236.41,14257.50,14236.41,14257.50
...,...,...,...,...,...
1881089,2025-05-02 19:59:10+00:00,20092.85,20099.47,20092.85,20099.47
1881090,2025-05-02 19:59:20+00:00,20100.19,20106.53,20100.19,20106.53
1881091,2025-05-02 19:59:30+00:00,20107.10,20110.80,20103.41,20110.80
1881092,2025-05-02 19:59:40+00:00,20111.43,20112.79,20103.82,20103.82


In [13]:
save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}_{price_source}.parquet"
df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")

Fichier sauvegardé sous le nom : ../marketData/NDX_10secs_20220214_to_20250502_TRADES.parquet
